# Wine Quality: Data Integration and PCA

This notebook contains the reproducible Wine Quality analysis owned by Nguyễn Hoàng Minh. The first stage loads the official UCI red and white wine files, verifies their schemas, adds a wine-type label, and integrates them without modifying the raw files.

## 1. Setup and source files

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RED_PATH = RAW_DIR / 'winequality-red.csv'
WHITE_PATH = RAW_DIR / 'winequality-white.csv'
INTEGRATED_PATH = PROCESSED_DIR / 'wine_quality_integrated.csv'

assert RED_PATH.exists(), f'Missing source file: {RED_PATH}'
assert WHITE_PATH.exists(), f'Missing source file: {WHITE_PATH}'

## 2. Load and verify the two source datasets

The UCI files use a semicolon (`;`) delimiter. Their schemas must match before row-wise integration.

In [2]:
red_raw = pd.read_csv(RED_PATH, sep=';')
white_raw = pd.read_csv(WHITE_PATH, sep=';')

assert list(red_raw.columns) == list(white_raw.columns), (
    'Red and white wine schemas do not match.'
)
assert red_raw.shape[1] == 12
assert white_raw.shape[1] == 12

source_summary = pd.DataFrame(
    {
        'dataset': ['red', 'white'],
        'rows': [len(red_raw), len(white_raw)],
        'columns_before_label': [red_raw.shape[1], white_raw.shape[1]],
        'missing_values': [
            int(red_raw.isna().sum().sum()),
            int(white_raw.isna().sum().sum()),
        ],
    }
)
source_summary

,dataset,rows,columns_before_label,missing_values
0,red,1599,12,0
1,white,4898,12,0


## 3. Add `wine_type` and integrate

The two files describe the same measurements and use the same schema, so row-wise concatenation is appropriate. The added `wine_type` field preserves the source of every observation and can later be used as a visualization label. It is not one of the physicochemical PCA inputs.

In [3]:
red_labeled = red_raw.assign(wine_type='red')
white_labeled = white_raw.assign(wine_type='white')

wine_integrated = pd.concat(
    [red_labeled, white_labeled],
    axis=0,
    ignore_index=True,
    verify_integrity=True,
)

expected_rows = len(red_raw) + len(white_raw)
assert len(wine_integrated) == expected_rows
assert wine_integrated.shape[1] == 13
assert set(wine_integrated['wine_type'].unique()) == {'red', 'white'}
assert wine_integrated['wine_type'].value_counts().to_dict() == {
    'white': len(white_raw),
    'red': len(red_raw),
}

integration_summary = pd.DataFrame(
    {
        'metric': [
            'red rows',
            'white rows',
            'integrated rows',
            'integrated columns',
            'integrated missing values',
        ],
        'value': [
            len(red_raw),
            len(white_raw),
            len(wine_integrated),
            wine_integrated.shape[1],
            int(wine_integrated.isna().sum().sum()),
        ],
    }
)
integration_summary

,metric,value
0,red rows,1599
1,white rows,4898
2,integrated rows,6497
3,integrated columns,13
4,integrated missing values,0


## 4. Save the reproducible integrated dataset

The processed file is generated from the two immutable raw files. It is excluded from Git and can be recreated by running this notebook.

In [4]:
wine_integrated.to_csv(INTEGRATED_PATH, index=False)

round_trip = pd.read_csv(INTEGRATED_PATH)
assert round_trip.shape == wine_integrated.shape
assert list(round_trip.columns) == list(wine_integrated.columns)

print(f'Saved integrated dataset to: {INTEGRATED_PATH}')
print(f'Final shape: {wine_integrated.shape}')
wine_integrated.head()

Saved integrated dataset to: D:\ml2\lab1_ml2\data\processed\wine_quality_integrated.csv
Final shape: (6497, 13)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


## 5. Data dictionary and feature classification

The dataset contains 11 physicochemical input features, one ordinal quality target, and one nominal integration label. The classification below explicitly distinguishes continuous/discrete, quantitative/qualitative, and numerical/categorical variables as required by the assignment.

In [5]:
PCA_FEATURES = [
    'fixed acidity',
    'volatile acidity',
    'citric acid',
    'residual sugar',
    'chlorides',
    'free sulfur dioxide',
    'total sulfur dioxide',
    'density',
    'pH',
    'sulphates',
    'alcohol',
]
TARGET_COLUMN = 'quality'
LABEL_COLUMN = 'wine_type'

expected_columns = PCA_FEATURES + [TARGET_COLUMN, LABEL_COLUMN]
assert list(wine_integrated.columns) == expected_columns
assert all(pd.api.types.is_numeric_dtype(wine_integrated[c]) for c in PCA_FEATURES)
assert pd.api.types.is_integer_dtype(wine_integrated[TARGET_COLUMN])
assert not pd.api.types.is_numeric_dtype(wine_integrated[LABEL_COLUMN])

In [6]:
feature_descriptions = {
    'fixed acidity': 'Concentration of non-volatile acids that contribute to wine acidity',
    'volatile acidity': 'Concentration of volatile acids, mainly acetic acid',
    'citric acid': 'Citric-acid concentration contributing freshness and flavor',
    'residual sugar': 'Sugar remaining after fermentation',
    'chlorides': 'Salt concentration in the wine',
    'free sulfur dioxide': 'Free SO2 available to protect the wine from oxidation and microbes',
    'total sulfur dioxide': 'Total free and bound SO2 in the wine',
    'density': 'Wine density relative to water',
    'pH': 'Acidity level expressed on the pH scale',
    'sulphates': 'Sulphate concentration associated with wine preservation',
    'alcohol': 'Alcohol content of the wine',
    'quality': 'Sensory quality score assigned to the wine',
    'wine_type': 'Source label identifying red or white wine',
}

rows = []
for column in wine_integrated.columns:
    is_pca_feature = column in PCA_FEATURES
    if is_pca_feature:
        role = 'input feature'
        value_nature = 'continuous'
        information_type = 'quantitative'
        representation = 'numerical'
    elif column == TARGET_COLUMN:
        role = 'target'
        value_nature = 'discrete (ordinal)'
        information_type = 'quantitative/ordinal'
        representation = 'numerical'
    else:
        role = 'integration/visualization label'
        value_nature = 'discrete (nominal)'
        information_type = 'qualitative'
        representation = 'categorical'

    rows.append(
        {
            'variable': column,
            'description': feature_descriptions[column],
            'role': role,
            'continuous_or_discrete': value_nature,
            'quantitative_or_qualitative': information_type,
            'numerical_or_categorical': representation,
            'actual_pandas_dtype': str(wine_integrated[column].dtype),
            'used_to_fit_pca': 'yes' if is_pca_feature else 'no',
        }
    )

feature_dictionary = pd.DataFrame(rows)
assert feature_dictionary.shape[0] == wine_integrated.shape[1]
assert feature_dictionary['used_to_fit_pca'].value_counts().to_dict() == {
    'yes': 11,
    'no': 2,
}
feature_dictionary

,variable,description,role,continuous_or_discrete,quantitative_or_qualitative,numerical_or_categorical,actual_pandas_dtype,used_to_fit_pca
0,fixed acidity,Concentration of non-volatile acids that contr...,input feature,continuous,quantitative,numerical,float64,yes
1,volatile acidity,"Concentration of volatile acids, mainly acetic...",input feature,continuous,quantitative,numerical,float64,yes
2,citric acid,Citric-acid concentration contributing freshne...,input feature,continuous,quantitative,numerical,float64,yes
3,residual sugar,Sugar remaining after fermentation,input feature,continuous,quantitative,numerical,float64,yes
4,chlorides,Salt concentration in the wine,input feature,continuous,quantitative,numerical,float64,yes
5,free sulfur dioxide,Free SO2 available to protect the wine from ox...,input feature,continuous,quantitative,numerical,float64,yes
6,total sulfur dioxide,Total free and bound SO2 in the wine,input feature,continuous,quantitative,numerical,float64,yes
7,density,Wine density relative to water,input feature,continuous,quantitative,numerical,float64,yes
8,pH,Acidity level expressed on the pH scale,input feature,continuous,quantitative,numerical,float64,yes
9,sulphates,Sulphate concentration associated with wine pr...,input feature,continuous,quantitative,numerical,float64,yes


In [7]:
TABLES_DIR = PROJECT_ROOT / 'report' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DICTIONARY_PATH = TABLES_DIR / 'wine_feature_dictionary.csv'
feature_dictionary.to_csv(FEATURE_DICTIONARY_PATH, index=False, encoding='utf-8-sig')

classification_summary = pd.DataFrame(
    {
        'group': ['PCA input features', 'target', 'integration label'],
        'count': [len(PCA_FEATURES), 1, 1],
        'variables': [', '.join(PCA_FEATURES), TARGET_COLUMN, LABEL_COLUMN],
    }
)

print(f'Saved feature dictionary to: {FEATURE_DICTIONARY_PATH}')
classification_summary

Saved feature dictionary to: D:\ml2\lab1_ml2\report\tables\wine_feature_dictionary.csv


,group,count,variables
0,PCA input features,11,"fixed acidity, volatile acidity, citric acid, ..."
1,target,1,quality
2,integration label,1,wine_type


## 6. Data-quality audit before cleaning

This stage detects and quantifies quality issues without changing the data. Duplicate observations and IQR outliers are flagged for a later cleaning decision rather than removed automatically. Because red and white wines have different distributions, outliers are calculated both globally and within each `wine_type`.

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

FIGURES_DIR = PROJECT_ROOT / 'report' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

numeric_columns = PCA_FEATURES + [TARGET_COLUMN]
missing_by_column = wine_integrated.isna().sum()
infinite_by_column = pd.Series(
    np.isinf(wine_integrated[numeric_columns].to_numpy()).sum(axis=0),
    index=numeric_columns,
    name='infinite_values',
)

duplicate_full_mask = wine_integrated.duplicated(keep='first')
duplicate_without_type_mask = wine_integrated.duplicated(
    subset=PCA_FEATURES + [TARGET_COLUMN],
    keep='first',
)
duplicate_by_type = (
    wine_integrated.groupby(LABEL_COLUMN, sort=True, group_keys=False)
    .apply(lambda group: int(group.duplicated(keep='first').sum()), include_groups=False)
    .rename('duplicate_rows')
)

print('Missing values by column:')
print(missing_by_column.to_string())
print('\nDuplicate rows by wine type:')
print(duplicate_by_type.to_string())

Missing values by column:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
wine_type               0

Duplicate rows by wine type:
wine_type
red      240
white    937


In [9]:
validity_checks = {
    'fixed acidity must be positive': wine_integrated['fixed acidity'] <= 0,
    'volatile acidity must be non-negative': wine_integrated['volatile acidity'] < 0,
    'citric acid must be non-negative': wine_integrated['citric acid'] < 0,
    'residual sugar must be non-negative': wine_integrated['residual sugar'] < 0,
    'chlorides must be non-negative': wine_integrated['chlorides'] < 0,
    'free sulfur dioxide must be non-negative': wine_integrated['free sulfur dioxide'] < 0,
    'total sulfur dioxide must be non-negative': wine_integrated['total sulfur dioxide'] < 0,
    'free sulfur dioxide cannot exceed total': (
        wine_integrated['free sulfur dioxide'] > wine_integrated['total sulfur dioxide']
    ),
    'density must be positive': wine_integrated['density'] <= 0,
    'pH must be between 0 and 14': ~wine_integrated['pH'].between(0, 14),
    'sulphates must be non-negative': wine_integrated['sulphates'] < 0,
    'alcohol percentage must be between 0 and 100': ~wine_integrated['alcohol'].between(0, 100),
    'quality must be between 0 and 10': ~wine_integrated[TARGET_COLUMN].between(0, 10),
    'wine_type must be red or white': ~wine_integrated[LABEL_COLUMN].isin(['red', 'white']),
}

invalid_rule_summary = pd.DataFrame(
    {
        'rule': list(validity_checks),
        'violation_count': [int(mask.sum()) for mask in validity_checks.values()],
    }
)
assert int(invalid_rule_summary['violation_count'].sum()) == 0
invalid_rule_summary

,rule,violation_count
0,fixed acidity must be positive,0
1,volatile acidity must be non-negative,0
2,citric acid must be non-negative,0
3,residual sugar must be non-negative,0
4,chlorides must be non-negative,0
5,free sulfur dioxide must be non-negative,0
6,total sulfur dioxide must be non-negative,0
7,free sulfur dioxide cannot exceed total,0
8,density must be positive,0
9,pH must be between 0 and 14,0


In [10]:
range_summary = wine_integrated[numeric_columns].agg(
    ['min', 'max', 'mean', 'median', 'std']
).T.reset_index(names='variable')
range_summary

,variable,min,max,mean,median,std
0,fixed acidity,3.80000,15.90000,7.215307,7.00000,1.296434
1,volatile acidity,0.08000,1.58000,0.339666,0.29000,0.164636
2,citric acid,0.00000,1.66000,0.318633,0.31000,0.145318
3,residual sugar,0.60000,65.80000,5.443235,3.00000,4.757804
4,chlorides,0.00900,0.61100,0.056034,0.04700,0.035034
5,free sulfur dioxide,1.00000,289.00000,30.525319,29.00000,17.749400
6,total sulfur dioxide,6.00000,440.00000,115.744574,118.00000,56.521855
7,density,0.98711,1.03898,0.994697,0.99489,0.002999
8,pH,2.72000,4.01000,3.218501,3.21000,0.160787
9,sulphates,0.22000,2.00000,0.531268,0.51000,0.148806


### IQR outlier detection

An IQR flag is a statistical warning, not proof of an invalid measurement. Type-aware thresholds are more appropriate for cleaning decisions because integrating red and white wines creates a mixture of different distributions.

In [11]:
def iqr_outlier_flags(frame, columns):
    q1 = frame[columns].quantile(0.25)
    q3 = frame[columns].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return frame[columns].lt(lower) | frame[columns].gt(upper)

global_outlier_flags = iqr_outlier_flags(wine_integrated, PCA_FEATURES)
type_outlier_flags = pd.DataFrame(
    False, index=wine_integrated.index, columns=PCA_FEATURES
)

for wine_type, group_index in wine_integrated.groupby(LABEL_COLUMN).groups.items():
    group = wine_integrated.loc[group_index]
    type_outlier_flags.loc[group_index, PCA_FEATURES] = iqr_outlier_flags(
        group, PCA_FEATURES
    ).to_numpy()

outlier_summary = pd.DataFrame(
    {
        'variable': PCA_FEATURES,
        'global_iqr_outliers': global_outlier_flags.sum().reindex(PCA_FEATURES).astype(int).values,
        'type_aware_iqr_outliers': type_outlier_flags.sum().reindex(PCA_FEATURES).astype(int).values,
    }
).sort_values('type_aware_iqr_outliers', ascending=False, ignore_index=True)

outlier_rows_global = int(global_outlier_flags.any(axis=1).sum())
outlier_rows_type_aware = int(type_outlier_flags.any(axis=1).sum())
outlier_summary

,variable,global_iqr_outliers,type_aware_iqr_outliers
0,chlorides,286,320
1,citric acid,509,271
2,volatile acidity,377,205
3,sulphates,191,183
4,fixed acidity,357,168
5,residual sugar,118,162
6,pH,73,110
7,free sulfur dioxide,62,80
8,total sulfur dioxide,10,74
9,density,3,50


In [12]:
audit_summary = pd.DataFrame(
    {
        'metric': [
            'rows',
            'columns',
            'missing cells',
            'infinite numerical cells',
            'duplicate excess rows (including wine_type)',
            'duplicate excess rows (ignoring wine_type)',
            'basic validity-rule violations',
            'global-IQR outlier cells',
            'global-IQR outlier rows',
            'type-aware IQR outlier cells',
            'type-aware IQR outlier rows',
        ],
        'value': [
            len(wine_integrated),
            wine_integrated.shape[1],
            int(missing_by_column.sum()),
            int(infinite_by_column.sum()),
            int(duplicate_full_mask.sum()),
            int(duplicate_without_type_mask.sum()),
            int(invalid_rule_summary['violation_count'].sum()),
            int(global_outlier_flags.sum().sum()),
            outlier_rows_global,
            int(type_outlier_flags.sum().sum()),
            outlier_rows_type_aware,
        ],
    }
)
audit_summary

,metric,value
0,rows,6497
1,columns,13
2,missing cells,0
3,infinite numerical cells,0
4,duplicate excess rows (including wine_type),1177
5,duplicate excess rows (ignoring wine_type),1179
6,basic validity-rule violations,0
7,global-IQR outlier cells,1989
8,global-IQR outlier rows,1473
9,type-aware IQR outlier cells,1636


In [13]:
AUDIT_SUMMARY_PATH = TABLES_DIR / 'wine_data_quality_summary.csv'
INVALID_RULES_PATH = TABLES_DIR / 'wine_invalid_rule_summary.csv'
RANGE_SUMMARY_PATH = TABLES_DIR / 'wine_range_summary.csv'
OUTLIER_SUMMARY_PATH = TABLES_DIR / 'wine_outlier_summary_iqr.csv'

audit_summary.to_csv(AUDIT_SUMMARY_PATH, index=False)
invalid_rule_summary.to_csv(INVALID_RULES_PATH, index=False)
range_summary.to_csv(RANGE_SUMMARY_PATH, index=False)
outlier_summary.to_csv(OUTLIER_SUMMARY_PATH, index=False)

print('Saved data-quality tables:')
for path in [AUDIT_SUMMARY_PATH, INVALID_RULES_PATH, RANGE_SUMMARY_PATH, OUTLIER_SUMMARY_PATH]:
    print(f'- {path}')

Saved data-quality tables:
- D:\ml2\lab1_ml2\report\tables\wine_data_quality_summary.csv
- D:\ml2\lab1_ml2\report\tables\wine_invalid_rule_summary.csv
- D:\ml2\lab1_ml2\report\tables\wine_range_summary.csv
- D:\ml2\lab1_ml2\report\tables\wine_outlier_summary_iqr.csv


In [14]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(4, 3, figsize=(15, 16))
axes = axes.flatten()

for axis, feature in zip(axes, PCA_FEATURES):
    sns.boxplot(
        data=wine_integrated,
        x=LABEL_COLUMN,
        y=feature,
        hue=LABEL_COLUMN,
        legend=False,
        palette={'red': '#b4443c', 'white': '#d7aa30'},
        ax=axis,
    )
    axis.set_title(feature)
    axis.set_xlabel('')

for axis in axes[len(PCA_FEATURES):]:
    axis.remove()

fig.suptitle('Wine Quality features before cleaning, grouped by wine type', fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.98))
BOXPLOT_PATH = FIGURES_DIR / 'wine_boxplots_before_cleaning.png'
fig.savefig(BOXPLOT_PATH, dpi=180, bbox_inches='tight')
plt.close(fig)
print(f'Saved boxplots to: {BOXPLOT_PATH}')

Saved boxplots to: D:\ml2\lab1_ml2\report\figures\wine_boxplots_before_cleaning.png


## 7. Cleaning decisions

The audit found no missing, infinite, or domain-invalid values. Exact duplicate rows within the same `wine_type` are removed so repeated observations do not receive extra weight in the statistical analysis or PCA. Records with the same measurements but different `wine_type` labels remain distinct. IQR outliers are retained because they satisfy the validity rules, occur frequently, and may represent genuine physicochemical variation. They are marked separately for transparent analysis.

In [15]:
wine_cleaned = wine_integrated.drop_duplicates(keep='first').reset_index(drop=True)
removed_duplicate_rows = len(wine_integrated) - len(wine_cleaned)

assert removed_duplicate_rows == int(duplicate_full_mask.sum())
assert removed_duplicate_rows == 1177
assert wine_cleaned.shape == (5320, 13)
assert int(wine_cleaned.duplicated(keep=False).sum()) == 0
assert int(wine_cleaned.isna().sum().sum()) == 0
assert list(wine_cleaned.columns) == list(wine_integrated.columns)

cleaning_by_type = (
    wine_integrated[LABEL_COLUMN]
    .value_counts()
    .rename('before')
    .to_frame()
    .join(wine_cleaned[LABEL_COLUMN].value_counts().rename('after'))
    .assign(removed=lambda frame: frame['before'] - frame['after'])
    .rename_axis(LABEL_COLUMN)
    .reset_index()
)
cleaning_by_type

,wine_type,before,after,removed
0,white,4898,3961,937
1,red,1599,1359,240


In [16]:
def evaluate_basic_validity(frame):
    checks = {
        'fixed acidity must be positive': frame['fixed acidity'] <= 0,
        'volatile acidity must be non-negative': frame['volatile acidity'] < 0,
        'citric acid must be non-negative': frame['citric acid'] < 0,
        'residual sugar must be non-negative': frame['residual sugar'] < 0,
        'chlorides must be non-negative': frame['chlorides'] < 0,
        'free sulfur dioxide must be non-negative': frame['free sulfur dioxide'] < 0,
        'total sulfur dioxide must be non-negative': frame['total sulfur dioxide'] < 0,
        'free sulfur dioxide cannot exceed total': (
            frame['free sulfur dioxide'] > frame['total sulfur dioxide']
        ),
        'density must be positive': frame['density'] <= 0,
        'pH must be between 0 and 14': ~frame['pH'].between(0, 14),
        'sulphates must be non-negative': frame['sulphates'] < 0,
        'alcohol percentage must be between 0 and 100': ~frame['alcohol'].between(0, 100),
        'quality must be between 0 and 10': ~frame[TARGET_COLUMN].between(0, 10),
        'wine_type must be red or white': ~frame[LABEL_COLUMN].isin(['red', 'white']),
    }
    return pd.DataFrame(
        {
            'rule': list(checks),
            'violation_count': [int(mask.sum()) for mask in checks.values()],
        }
    )

cleaned_invalid_rule_summary = evaluate_basic_validity(wine_cleaned)
cleaned_infinite_cells = int(
    np.isinf(wine_cleaned[numeric_columns].to_numpy()).sum()
)
assert int(cleaned_invalid_rule_summary['violation_count'].sum()) == 0
assert cleaned_infinite_cells == 0
cleaned_invalid_rule_summary

,rule,violation_count
0,fixed acidity must be positive,0
1,volatile acidity must be non-negative,0
2,citric acid must be non-negative,0
3,residual sugar must be non-negative,0
4,chlorides must be non-negative,0
5,free sulfur dioxide must be non-negative,0
6,total sulfur dioxide must be non-negative,0
7,free sulfur dioxide cannot exceed total,0
8,density must be positive,0
9,pH must be between 0 and 14,0


In [17]:
cleaned_type_outlier_flags = pd.DataFrame(
    False, index=wine_cleaned.index, columns=PCA_FEATURES
)

for wine_type, group_index in wine_cleaned.groupby(LABEL_COLUMN).groups.items():
    group = wine_cleaned.loc[group_index]
    cleaned_type_outlier_flags.loc[group_index, PCA_FEATURES] = iqr_outlier_flags(
        group, PCA_FEATURES
    ).to_numpy()

cleaned_has_iqr_outlier = cleaned_type_outlier_flags.any(axis=1)
cleaned_outlier_summary = pd.DataFrame(
    {
        'variable': PCA_FEATURES,
        'type_aware_iqr_outliers_after_cleaning': (
            cleaned_type_outlier_flags.sum().reindex(PCA_FEATURES).astype(int).values
        ),
    }
).sort_values(
    'type_aware_iqr_outliers_after_cleaning', ascending=False, ignore_index=True
)

cleaned_outlier_rows = int(cleaned_has_iqr_outlier.sum())
cleaned_outlier_cells = int(cleaned_type_outlier_flags.sum().sum())
cleaned_outlier_summary

,variable,type_aware_iqr_outliers_after_cleaning
0,chlorides,265
1,citric acid,224
2,volatile acidity,152
3,sulphates,151
4,fixed acidity,147
5,residual sugar,142
6,pH,74
7,free sulfur dioxide,70
8,total sulfur dioxide,59
9,density,41


In [18]:
before_after_cleaning = pd.DataFrame(
    {
        'metric': [
            'rows',
            'columns',
            'missing cells',
            'infinite numerical cells',
            'duplicate excess rows',
            'basic validity-rule violations',
            'type-aware IQR outlier cells',
            'type-aware IQR outlier rows',
            'type-aware IQR outlier row rate (%)',
        ],
        'before': [
            len(wine_integrated),
            wine_integrated.shape[1],
            int(missing_by_column.sum()),
            int(infinite_by_column.sum()),
            int(duplicate_full_mask.sum()),
            int(invalid_rule_summary['violation_count'].sum()),
            int(type_outlier_flags.sum().sum()),
            outlier_rows_type_aware,
            round(100 * outlier_rows_type_aware / len(wine_integrated), 2),
        ],
        'after': [
            len(wine_cleaned),
            wine_cleaned.shape[1],
            int(wine_cleaned.isna().sum().sum()),
            cleaned_infinite_cells,
            int(wine_cleaned.duplicated().sum()),
            int(cleaned_invalid_rule_summary['violation_count'].sum()),
            cleaned_outlier_cells,
            cleaned_outlier_rows,
            round(100 * cleaned_outlier_rows / len(wine_cleaned), 2),
        ],
    }
)
before_after_cleaning['change'] = (
    before_after_cleaning['after'] - before_after_cleaning['before']
)
before_after_cleaning

,metric,before,after,change
0,rows,6497.00,5320.00,-1177.00
1,columns,13.00,13.00,0.00
2,missing cells,0.00,0.00,0.00
3,infinite numerical cells,0.00,0.00,0.00
4,duplicate excess rows,1177.00,0.00,-1177.00
5,basic validity-rule violations,0.00,0.00,0.00
6,type-aware IQR outlier cells,1636.00,1337.00,-299.00
7,type-aware IQR outlier rows,1288.00,1046.00,-242.00
8,type-aware IQR outlier row rate (%),19.82,19.66,-0.16


In [19]:
cleaning_decisions = pd.DataFrame(
    {
        'issue': ['missing values', 'infinite/invalid values', 'exact duplicates', 'IQR outliers'],
        'finding': [
            'None detected',
            'None detected by the documented validity rules',
            f'{removed_duplicate_rows} excess rows detected within wine_type',
            f'{cleaned_outlier_rows} cleaned rows flagged after type-aware IQR recalculation',
        ],
        'action': [
            'No imputation required',
            'No correction required',
            'Remove exact duplicates and retain the first occurrence',
            'Retain valid observations; store a separate flag for transparency',
        ],
        'reason': [
            'The source files are complete',
            'All observations satisfy the basic physical/domain checks',
            'Repeated rows would receive extra weight in statistics and PCA',
            'IQR flags are frequent and do not prove measurement error',
        ],
    }
)
cleaning_decisions

,issue,finding,action,reason
0,missing values,None detected,No imputation required,The source files are complete
1,infinite/invalid values,None detected by the documented validity rules,No correction required,All observations satisfy the basic physical/do...
2,exact duplicates,1177 excess rows detected within wine_type,Remove exact duplicates and retain the first o...,Repeated rows would receive extra weight in st...
3,IQR outliers,1046 cleaned rows flagged after type-aware IQR...,Retain valid observations; store a separate fl...,IQR flags are frequent and do not prove measur...


In [20]:
CLEANED_PATH = PROCESSED_DIR / 'wine_quality_cleaned.csv'
OUTLIER_FLAGS_PATH = PROCESSED_DIR / 'wine_quality_cleaned_outlier_flags.csv'
BEFORE_AFTER_PATH = TABLES_DIR / 'wine_cleaning_before_after.csv'
CLEANING_DECISIONS_PATH = TABLES_DIR / 'wine_cleaning_decisions.csv'
CLEANING_BY_TYPE_PATH = TABLES_DIR / 'wine_cleaning_by_type.csv'
CLEANED_OUTLIER_SUMMARY_PATH = TABLES_DIR / 'wine_outlier_summary_after_cleaning.csv'

wine_cleaned.to_csv(CLEANED_PATH, index=False)
outlier_flags_export = cleaned_type_outlier_flags.copy()
outlier_flags_export.insert(0, 'cleaned_row_id', wine_cleaned.index)
outlier_flags_export['has_iqr_outlier'] = cleaned_has_iqr_outlier.to_numpy()
outlier_flags_export.to_csv(OUTLIER_FLAGS_PATH, index=False)
before_after_cleaning.to_csv(BEFORE_AFTER_PATH, index=False)
cleaning_decisions.to_csv(CLEANING_DECISIONS_PATH, index=False)
cleaning_by_type.to_csv(CLEANING_BY_TYPE_PATH, index=False)
cleaned_outlier_summary.to_csv(CLEANED_OUTLIER_SUMMARY_PATH, index=False)

cleaned_round_trip = pd.read_csv(CLEANED_PATH)
assert cleaned_round_trip.shape == wine_cleaned.shape
assert int(cleaned_round_trip.duplicated().sum()) == 0

print(f'Saved cleaned dataset to: {CLEANED_PATH}')
print(f'Rows before cleaning: {len(wine_integrated)}')
print(f'Rows after cleaning: {len(wine_cleaned)}')
print(f'Exact duplicate rows removed: {removed_duplicate_rows}')
print(f'Valid IQR-outlier rows retained and flagged: {cleaned_outlier_rows}')

Saved cleaned dataset to: D:\ml2\lab1_ml2\data\processed\wine_quality_cleaned.csv
Rows before cleaning: 6497
Rows after cleaning: 5320
Exact duplicate rows removed: 1177
Valid IQR-outlier rows retained and flagged: 1046


## 8. Descriptive statistics and categorical analysis

All statistics in this section use the cleaned dataset. The 11 physicochemical features are summarized numerically. `quality` is retained as an ordinal target, while `wine_type` is analyzed as a nominal categorical label.

In [21]:
descriptive_statistics = pd.DataFrame(
    {
        'count': wine_cleaned[PCA_FEATURES].count(),
        'mean': wine_cleaned[PCA_FEATURES].mean(),
        'sample_variance': wine_cleaned[PCA_FEATURES].var(ddof=1),
        'sample_std': wine_cleaned[PCA_FEATURES].std(ddof=1),
        'min': wine_cleaned[PCA_FEATURES].min(),
        'q1': wine_cleaned[PCA_FEATURES].quantile(0.25),
        'median': wine_cleaned[PCA_FEATURES].median(),
        'q3': wine_cleaned[PCA_FEATURES].quantile(0.75),
        'max': wine_cleaned[PCA_FEATURES].max(),
    }
).rename_axis('variable').reset_index()

assert descriptive_statistics.shape == (11, 10)
assert descriptive_statistics['count'].eq(len(wine_cleaned)).all()
descriptive_statistics

,variable,count,mean,sample_variance,sample_std,min,q1,median,q3,max
0,fixed acidity,5320,7.215179,1.741531,1.319671,3.80000,6.4000,7.00000,7.70000,15.90000
1,volatile acidity,5320,0.344130,0.028307,0.168248,0.08000,0.2300,0.30000,0.41000,1.58000
2,citric acid,5320,0.318494,0.021655,0.147157,0.00000,0.2400,0.31000,0.40000,1.66000
3,residual sugar,5320,5.048477,20.251621,4.500180,0.60000,1.8000,2.70000,7.50000,65.80000
4,chlorides,5320,0.056690,0.001359,0.036863,0.00900,0.0380,0.04700,0.06600,0.61100
5,free sulfur dioxide,5320,30.036654,317.019619,17.805045,1.00000,16.0000,28.00000,41.00000,289.00000
6,total sulfur dioxide,5320,114.109023,3223.312421,56.774223,6.00000,74.0000,116.00000,153.25000,440.00000
7,density,5320,0.994535,0.000009,0.002966,0.98711,0.9922,0.99465,0.99677,1.03898
8,pH,5320,3.224664,0.025721,0.160379,2.72000,3.1100,3.21000,3.33000,4.01000
9,sulphates,5320,0.533357,0.022423,0.149743,0.22000,0.4300,0.51000,0.60000,2.00000


In [22]:
covariance_matrix = wine_cleaned[PCA_FEATURES].cov(ddof=1)
assert covariance_matrix.shape == (11, 11)
assert np.allclose(covariance_matrix, covariance_matrix.T)

by_type_mean_variance = wine_cleaned.groupby(LABEL_COLUMN)[PCA_FEATURES].agg(
    ['mean', 'var']
)
by_type_mean_variance.columns = [
    f'{feature}_{statistic}' for feature, statistic in by_type_mean_variance.columns
]

covariance_matrix

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
fixed acidity,1.741531,0.047682,0.064149,-0.620235,0.014055,-6.616452,-24.535195,0.001871,-0.057395,0.060241,-0.160532
volatile acidity,0.047682,0.028307,-0.009512,-0.123942,0.002280,-1.045605,-3.827704,0.000154,0.006656,0.005738,-0.013052
citric acid,0.064149,-0.009512,0.021655,0.096811,0.000299,0.342843,1.627795,0.000041,-0.008136,0.001304,-0.000959
residual sugar,-0.620235,-0.123942,0.096811,20.251621,-0.020420,31.947588,124.558297,0.006952,-0.169263,-0.117736,-1.629050
chlorides,0.014055,0.002280,0.000299,-0.020420,0.001359,-0.122485,-0.564696,0.000041,0.000153,0.002236,-0.011786
free sulfur dioxide,-6.616452,-1.045605,0.342843,31.947588,-0.122485,317.019619,728.318432,0.000326,-0.404768,-0.528553,-3.589903
total sulfur dioxide,-24.535195,-3.827704,1.627795,124.558297,-0.564696,728.318432,3223.312421,0.001130,-2.030109,-2.345034,-16.765542
density,0.001871,0.000154,0.000041,0.006952,0.000041,0.000326,0.001130,0.000009,0.000016,0.000126,-0.002349
pH,-0.057395,0.006656,-0.008136,-0.169263,0.000153,-0.404768,-2.030109,0.000016,0.025721,0.004038,0.018509
sulphates,0.060241,0.005738,0.001304,-0.117736,0.002236,-0.528553,-2.345034,0.000126,0.004038,0.022423,-0.003060


In [23]:
wine_type_frequency = (
    wine_cleaned[LABEL_COLUMN]
    .value_counts()
    .sort_index()
    .rename('count')
    .to_frame()
)
wine_type_frequency['percentage'] = (
    100 * wine_type_frequency['count'] / len(wine_cleaned)
).round(2)

quality_frequency = (
    wine_cleaned[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename('count')
    .to_frame()
)
quality_frequency['percentage'] = (
    100 * quality_frequency['count'] / len(wine_cleaned)
).round(2)

quality_by_type_counts = pd.crosstab(
    wine_cleaned[LABEL_COLUMN], wine_cleaned[TARGET_COLUMN]
)
quality_by_type_row_percent = (
    100 * quality_by_type_counts.div(quality_by_type_counts.sum(axis=1), axis=0)
).round(2)

print('Wine-type frequency:')
print(wine_type_frequency.to_string())
print('\nQuality frequency:')
print(quality_frequency.to_string())
print('\nQuality by wine type:')
print(quality_by_type_counts.to_string())

Wine-type frequency:
           count  percentage
wine_type                   
red         1359       25.55
white       3961       74.45

Quality frequency:
         count  percentage
quality                   
3           30        0.56
4          206        3.87
5         1752       32.93
6         2323       43.67
7          856       16.09
8          148        2.78
9            5        0.09

Quality by wine type:
quality     3    4     5     6    7    8  9
wine_type                                  
red        10   53   577   535  167   17  0
white      20  153  1175  1788  689  131  5


In [24]:
from scipy.stats import chi2_contingency

chi2_statistic, chi2_p_value, chi2_dof, expected_counts = chi2_contingency(
    quality_by_type_counts
)
expected_counts = pd.DataFrame(
    expected_counts,
    index=quality_by_type_counts.index,
    columns=quality_by_type_counts.columns,
)

n_observations = int(quality_by_type_counts.to_numpy().sum())
n_rows, n_columns = quality_by_type_counts.shape
phi_squared = chi2_statistic / n_observations
phi_squared_corrected = max(
    0,
    phi_squared - ((n_columns - 1) * (n_rows - 1)) / (n_observations - 1),
)
rows_corrected = n_rows - ((n_rows - 1) ** 2) / (n_observations - 1)
columns_corrected = n_columns - ((n_columns - 1) ** 2) / (n_observations - 1)
cramers_v_corrected = np.sqrt(
    phi_squared_corrected / min(columns_corrected - 1, rows_corrected - 1)
)

expected_below_five = int((expected_counts < 5).to_numpy().sum())
expected_below_five_percentage = 100 * expected_below_five / expected_counts.size
minimum_expected_count = float(expected_counts.to_numpy().min())

categorical_association = pd.DataFrame(
    {
        'measure': [
            'chi-square statistic',
            'degrees of freedom',
            'p-value',
            'bias-corrected Cramers V',
            'minimum expected count',
            'expected cells below 5',
            'expected cells below 5 (%)',
        ],
        'value': [
            chi2_statistic,
            chi2_dof,
            chi2_p_value,
            cramers_v_corrected,
            minimum_expected_count,
            expected_below_five,
            expected_below_five_percentage,
        ],
    }
)
categorical_association

,measure,value
0,chi-square statistic,9.246242e+01
1,degrees of freedom,6.000000e+00
2,p-value,9.325320e-18
3,bias-corrected Cramers V,1.274958e-01
4,minimum expected count,1.277256e+00
5,expected cells below 5,2.000000e+00
6,expected cells below 5 (%),1.428571e+01


### Interpretation of the categorical association

The Chi-square test treats the ordinal `quality` score as categories and tests whether its distribution is independent of `wine_type`; it does not use the ordering of quality scores. A statistically significant result should be interpreted together with Cramér's V, which measures association strength. Rare quality categories also require checking the expected-count assumptions.

In [25]:
DESCRIPTIVE_PATH = TABLES_DIR / 'wine_descriptive_statistics.csv'
COVARIANCE_PATH = TABLES_DIR / 'wine_covariance_matrix.csv'
BY_TYPE_STATS_PATH = TABLES_DIR / 'wine_descriptive_by_type_mean_variance.csv'
WINE_TYPE_FREQUENCY_PATH = TABLES_DIR / 'wine_type_frequency.csv'
QUALITY_FREQUENCY_PATH = TABLES_DIR / 'wine_quality_frequency.csv'
CONTINGENCY_COUNTS_PATH = TABLES_DIR / 'wine_quality_by_type_counts.csv'
CONTINGENCY_PERCENT_PATH = TABLES_DIR / 'wine_quality_by_type_row_percent.csv'
EXPECTED_COUNTS_PATH = TABLES_DIR / 'wine_quality_by_type_expected_counts.csv'
CATEGORICAL_ASSOCIATION_PATH = TABLES_DIR / 'wine_categorical_association.csv'

descriptive_statistics.to_csv(DESCRIPTIVE_PATH, index=False)
covariance_matrix.to_csv(COVARIANCE_PATH)
by_type_mean_variance.to_csv(BY_TYPE_STATS_PATH)
wine_type_frequency.to_csv(WINE_TYPE_FREQUENCY_PATH)
quality_frequency.to_csv(QUALITY_FREQUENCY_PATH)
quality_by_type_counts.to_csv(CONTINGENCY_COUNTS_PATH)
quality_by_type_row_percent.to_csv(CONTINGENCY_PERCENT_PATH)
expected_counts.to_csv(EXPECTED_COUNTS_PATH)
categorical_association.to_csv(CATEGORICAL_ASSOCIATION_PATH, index=False)

print(f'Chi-square = {chi2_statistic:.4f}, df = {chi2_dof}, p = {chi2_p_value:.4e}')
print(f'Bias-corrected Cramers V = {cramers_v_corrected:.4f}')
print(
    f'Expected-count check: minimum={minimum_expected_count:.3f}; '
    f'{expected_below_five}/{expected_counts.size} cells below 5 '
    f'({expected_below_five_percentage:.2f}%).'
)

Chi-square = 92.4624, df = 6, p = 9.3253e-18
Bias-corrected Cramers V = 0.1275
Expected-count check: minimum=1.277; 2/14 cells below 5 (14.29%).


In [26]:
fig, axis = plt.subplots(figsize=(10, 6))
quality_by_type_row_percent.T.plot(
    kind='bar',
    color={'red': '#b4443c', 'white': '#d7aa30'},
    ax=axis,
)
axis.set_title('Quality-score distribution within each wine type')
axis.set_xlabel('Quality score')
axis.set_ylabel('Percentage within wine type')
axis.legend(title='Wine type')
axis.tick_params(axis='x', rotation=0)
fig.tight_layout()
QUALITY_DISTRIBUTION_PATH = FIGURES_DIR / 'wine_quality_distribution_by_type.png'
fig.savefig(QUALITY_DISTRIBUTION_PATH, dpi=180, bbox_inches='tight')
plt.close(fig)
print(f'Saved quality distribution figure to: {QUALITY_DISTRIBUTION_PATH}')

Saved quality distribution figure to: D:\ml2\lab1_ml2\report\figures\wine_quality_distribution_by_type.png


## 9. Correlation analysis

Pearson correlation is calculated for the 11 continuous physicochemical PCA features in the cleaned dataset. The diagonal is excluded when ranking feature pairs. `quality` and `wine_type` are not included in this feature-correlation matrix.

In [27]:
correlation_matrix = wine_cleaned[PCA_FEATURES].corr(method='pearson')

assert correlation_matrix.shape == (11, 11)
assert not correlation_matrix.isna().any().any()
assert np.allclose(correlation_matrix, correlation_matrix.T)
assert np.allclose(np.diag(correlation_matrix), 1.0)

upper_triangle_mask = np.triu(
    np.ones(correlation_matrix.shape, dtype=bool), k=1
)
correlation_pairs = (
    correlation_matrix
    .where(upper_triangle_mask)
    .stack()
    .dropna()
    .rename('pearson_r')
    .reset_index()
    .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2'})
)
correlation_pairs['absolute_r'] = correlation_pairs['pearson_r'].abs()
correlation_pairs = correlation_pairs.sort_values(
    ['absolute_r', 'feature_1', 'feature_2'],
    ascending=[False, True, True],
).reset_index(drop=True)
correlation_pairs.insert(0, 'rank', np.arange(1, len(correlation_pairs) + 1))

expected_pair_count = len(PCA_FEATURES) * (len(PCA_FEATURES) - 1) // 2
assert len(correlation_pairs) == expected_pair_count == 55
assert correlation_pairs['absolute_r'].is_monotonic_decreasing

strongest_overall_pair = correlation_pairs.iloc[0]
correlation_pairs.head(10)

,rank,feature_1,feature_2,pearson_r,absolute_r
0,1,free sulfur dioxide,total sulfur dioxide,0.720488,0.720488
1,2,density,alcohol,-0.667811,0.667811
2,3,residual sugar,density,0.520930,0.520930
3,4,residual sugar,total sulfur dioxide,0.487519,0.487519
4,5,fixed acidity,density,0.478180,0.478180
5,6,chlorides,sulphates,0.405051,0.405051
6,7,volatile acidity,total sulfur dioxide,-0.400716,0.400716
7,8,residual sugar,free sulfur dioxide,0.398717,0.398717
8,9,volatile acidity,citric acid,-0.384192,0.384192
9,10,chlorides,density,0.371867,0.371867


In [28]:
correlation_scope_rows = [
    {
        'scope': 'overall',
        'sample_size': len(wine_cleaned),
        'feature_1': strongest_overall_pair['feature_1'],
        'feature_2': strongest_overall_pair['feature_2'],
        'pearson_r': strongest_overall_pair['pearson_r'],
        'absolute_r': strongest_overall_pair['absolute_r'],
    }
]

for wine_type, group in wine_cleaned.groupby(LABEL_COLUMN, sort=True):
    type_correlation = group[PCA_FEATURES].corr(method='pearson')
    type_pairs = (
        type_correlation
        .where(upper_triangle_mask)
        .stack()
        .dropna()
        .rename('pearson_r')
        .reset_index()
        .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2'})
    )
    type_pairs['absolute_r'] = type_pairs['pearson_r'].abs()
    strongest_type_pair = type_pairs.sort_values(
        ['absolute_r', 'feature_1', 'feature_2'],
        ascending=[False, True, True],
    ).iloc[0]
    correlation_scope_rows.append(
        {
            'scope': wine_type,
            'sample_size': len(group),
            'feature_1': strongest_type_pair['feature_1'],
            'feature_2': strongest_type_pair['feature_2'],
            'pearson_r': strongest_type_pair['pearson_r'],
            'absolute_r': strongest_type_pair['absolute_r'],
        }
    )

strongest_correlation_by_scope = pd.DataFrame(correlation_scope_rows)
print(
    'Strongest overall pair: '
    f"{strongest_overall_pair['feature_1']} and "
    f"{strongest_overall_pair['feature_2']} "
    f"(r = {strongest_overall_pair['pearson_r']:.4f})"
)
strongest_correlation_by_scope

Strongest overall pair: free sulfur dioxide and total sulfur dioxide (r = 0.7205)


,scope,sample_size,feature_1,feature_2,pearson_r,absolute_r
0,overall,5320,free sulfur dioxide,total sulfur dioxide,0.720488,0.720488
1,red,1359,fixed acidity,pH,-0.686685,0.686685
2,white,3961,residual sugar,density,0.820498,0.820498


### Interpretation note

Pearson's coefficient measures linear association, not causation. The overall matrix pools red and white wines, so the strongest relationship within either wine type can differ from the strongest relationship in the integrated dataset. The by-scope comparison is retained to make that integration effect visible.

In [29]:
CORRELATION_MATRIX_PATH = TABLES_DIR / 'wine_correlation_matrix.csv'
CORRELATION_PAIRS_PATH = TABLES_DIR / 'wine_correlation_pairs_ranked.csv'
CORRELATION_SCOPE_PATH = TABLES_DIR / 'wine_strongest_correlation_by_scope.csv'

correlation_matrix.to_csv(CORRELATION_MATRIX_PATH)
correlation_pairs.to_csv(CORRELATION_PAIRS_PATH, index=False)
strongest_correlation_by_scope.to_csv(CORRELATION_SCOPE_PATH, index=False)

print(f'Saved correlation matrix to: {CORRELATION_MATRIX_PATH}')
print(f'Saved ranked correlation pairs to: {CORRELATION_PAIRS_PATH}')
print(f'Saved by-scope comparison to: {CORRELATION_SCOPE_PATH}')

Saved correlation matrix to: D:\ml2\lab1_ml2\report\tables\wine_correlation_matrix.csv
Saved ranked correlation pairs to: D:\ml2\lab1_ml2\report\tables\wine_correlation_pairs_ranked.csv
Saved by-scope comparison to: D:\ml2\lab1_ml2\report\tables\wine_strongest_correlation_by_scope.csv


In [30]:
heatmap_mask = np.triu(np.ones_like(correlation_matrix, dtype=bool), k=1)
fig, axis = plt.subplots(figsize=(12, 9))
sns.heatmap(
    correlation_matrix,
    mask=heatmap_mask,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0,
    annot=True,
    fmt='.2f',
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
    ax=axis,
)
axis.grid(False)
axis.set_title('Pearson correlation among cleaned Wine Quality features', pad=14)
axis.tick_params(axis='x', rotation=45)
axis.tick_params(axis='y', rotation=0)
fig.tight_layout()
CORRELATION_HEATMAP_PATH = FIGURES_DIR / 'wine_correlation_heatmap.png'
fig.savefig(CORRELATION_HEATMAP_PATH, dpi=180, bbox_inches='tight')
plt.close(fig)
print(f'Saved correlation heatmap to: {CORRELATION_HEATMAP_PATH}')

Saved correlation heatmap to: D:\ml2\lab1_ml2\report\figures\wine_correlation_heatmap.png


## 10. Standardization before PCA

The 11 PCA features use very different units and numerical scales. PCA based on unscaled covariance would therefore give disproportionate weight to high-variance measurements such as sulfur dioxide. Each cleaned feature is transformed with the z-score $z=(x-\mu)/\sigma$ using `StandardScaler`. The ordinal target `quality` and nominal label `wine_type` are carried forward unchanged and are not used to fit the scaler.

In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=True, with_std=True)
standardized_array = scaler.fit_transform(wine_cleaned[PCA_FEATURES])
standardized_features = pd.DataFrame(
    standardized_array,
    columns=PCA_FEATURES,
    index=wine_cleaned.index,
)

assert standardized_features.shape == (len(wine_cleaned), len(PCA_FEATURES))
assert np.isfinite(standardized_features.to_numpy()).all()
assert np.allclose(standardized_features.mean().to_numpy(), 0.0, atol=1e-12)
assert np.allclose(
    standardized_features.std(ddof=0).to_numpy(), 1.0, atol=1e-12
)

wine_standardized = standardized_features.copy()
wine_standardized[TARGET_COLUMN] = wine_cleaned[TARGET_COLUMN].to_numpy()
wine_standardized[LABEL_COLUMN] = wine_cleaned[LABEL_COLUMN].to_numpy()
assert list(wine_standardized.columns) == list(wine_cleaned.columns)
assert wine_standardized[TARGET_COLUMN].equals(wine_cleaned[TARGET_COLUMN])
assert wine_standardized[LABEL_COLUMN].equals(wine_cleaned[LABEL_COLUMN])

standardization_check = pd.DataFrame(
    {
        'variable': PCA_FEATURES,
        'original_mean': wine_cleaned[PCA_FEATURES].mean().to_numpy(),
        'original_population_std': wine_cleaned[PCA_FEATURES].std(ddof=0).to_numpy(),
        'scaler_center': scaler.mean_,
        'scaler_scale': scaler.scale_,
        'standardized_mean': standardized_features.mean().to_numpy(),
        'standardized_population_std': standardized_features.std(ddof=0).to_numpy(),
        'minimum_z': standardized_features.min().to_numpy(),
        'maximum_z': standardized_features.max().to_numpy(),
        'maximum_absolute_z': standardized_features.abs().max().to_numpy(),
        'count_absolute_z_above_3': (standardized_features.abs() > 3).sum().to_numpy(),
        'count_absolute_z_above_5': (standardized_features.abs() > 5).sum().to_numpy(),
    }
)

standardization_check

,variable,original_mean,original_population_std,scaler_center,scaler_scale,standardized_mean,standardized_population_std,minimum_z,maximum_z,maximum_absolute_z,count_absolute_z_above_3,count_absolute_z_above_5
0,fixed acidity,7.215179,1.319547,7.215179,1.319547,3.205456e-16,1.0,-2.588145,6.581671,6.581671,108,8
1,volatile acidity,0.344130,0.168232,0.344130,0.168232,-6.410912e-17,1.0,-1.570028,7.346207,7.346207,77,4
2,citric acid,0.318494,0.147144,0.318494,0.147144,2.671213e-17,1.0,-2.164515,9.116989,9.116989,26,2
3,residual sugar,5.048477,4.499757,5.048477,4.499757,0.000000e+00,1.0,-0.988604,13.501067,13.501067,42,2
4,chlorides,0.056690,0.036860,0.056690,0.036860,1.282182e-16,1.0,-1.293816,15.038318,15.038318,94,33
5,free sulfur dioxide,30.036654,17.803371,30.036654,17.803371,2.136971e-17,1.0,-1.630964,14.545748,14.545748,27,7
6,total sulfur dioxide,114.109023,56.768887,114.109023,56.768887,-8.547883e-17,1.0,-1.904371,5.740662,5.740662,8,1
7,density,0.994535,0.002965,0.994535,0.002965,1.857027e-14,1.0,-2.504125,14.988637,14.988637,3,2
8,pH,3.224664,0.160364,3.224664,0.160364,6.197215e-16,1.0,-3.146985,4.897208,4.897208,32,0
9,sulphates,0.533357,0.149729,0.533357,0.149729,-2.136971e-17,1.0,-2.092831,9.795325,9.795325,63,13


In [32]:
rows_with_absolute_z_above_3 = int(
    (standardized_features.abs() > 3).any(axis=1).sum()
)
rows_with_absolute_z_above_5 = int(
    (standardized_features.abs() > 5).any(axis=1).sum()
)
values_with_absolute_z_above_3 = int(
    (standardized_features.abs() > 3).to_numpy().sum()
)
values_with_absolute_z_above_5 = int(
    (standardized_features.abs() > 5).to_numpy().sum()
)

standardization_summary = pd.DataFrame(
    {
        'metric': [
            'rows',
            'features standardized',
            'maximum absolute standardized mean',
            'maximum absolute deviation of population std from 1',
            'rows containing at least one absolute z-score above 3',
            'values with absolute z-score above 3',
            'rows containing at least one absolute z-score above 5',
            'values with absolute z-score above 5',
        ],
        'value': [
            len(standardized_features),
            standardized_features.shape[1],
            standardized_features.mean().abs().max(),
            (standardized_features.std(ddof=0) - 1).abs().max(),
            rows_with_absolute_z_above_3,
            values_with_absolute_z_above_3,
            rows_with_absolute_z_above_5,
            values_with_absolute_z_above_5,
        ],
    }
)
standardization_summary

,metric,value
0,rows,5.320000e+03
1,features standardized,1.100000e+01
2,maximum absolute standardized mean,1.857027e-14
3,maximum absolute deviation of population std f...,2.220446e-16
4,rows containing at least one absolute z-score ...,4.310000e+02
5,values with absolute z-score above 3,4.820000e+02
6,rows containing at least one absolute z-score ...,6.300000e+01
7,values with absolute z-score above 5,7.200000e+01


### Standardization notes

`StandardScaler` uses the population standard deviation (`ddof=0`), so each transformed feature is checked against a population standard deviation of 1. Standardization changes location and scale but does not remove skewness or outliers. The large absolute z-scores are retained because the preceding cleaning stage found no evidence that those observations were invalid. For this descriptive PCA, the scaler is fitted on the complete cleaned dataset; a predictive train/test workflow would instead fit preprocessing only on the training set.

In [33]:
STANDARDIZED_PATH = PROCESSED_DIR / 'wine_quality_standardized.csv'
STANDARDIZATION_CHECK_PATH = TABLES_DIR / 'wine_standardization_check.csv'
STANDARDIZATION_SUMMARY_PATH = TABLES_DIR / 'wine_standardization_summary.csv'

wine_standardized.to_csv(STANDARDIZED_PATH, index=False)
standardization_check.to_csv(STANDARDIZATION_CHECK_PATH, index=False)
standardization_summary.to_csv(STANDARDIZATION_SUMMARY_PATH, index=False)

standardized_round_trip = pd.read_csv(STANDARDIZED_PATH)
assert standardized_round_trip.shape == wine_standardized.shape
assert standardized_round_trip[TARGET_COLUMN].equals(wine_cleaned[TARGET_COLUMN])
assert standardized_round_trip[LABEL_COLUMN].equals(wine_cleaned[LABEL_COLUMN])
assert np.allclose(
    standardized_round_trip[PCA_FEATURES].to_numpy(),
    standardized_features.to_numpy(),
)

print(f'Saved standardized dataset to: {STANDARDIZED_PATH}')
print(f'Saved standardization checks to: {STANDARDIZATION_CHECK_PATH}')
print(f'Saved standardization summary to: {STANDARDIZATION_SUMMARY_PATH}')
print(
    f'Rows with at least one |z| > 3: {rows_with_absolute_z_above_3}; '
    f'rows with at least one |z| > 5: {rows_with_absolute_z_above_5}.'
)

Saved standardized dataset to: D:\ml2\lab1_ml2\data\processed\wine_quality_standardized.csv
Saved standardization checks to: D:\ml2\lab1_ml2\report\tables\wine_standardization_check.csv
Saved standardization summary to: D:\ml2\lab1_ml2\report\tables\wine_standardization_summary.csv
Rows with at least one |z| > 3: 431; rows with at least one |z| > 5: 63.


## 11. PCA fit and explained variance

PCA is fitted to all 11 standardized physicochemical features with the full deterministic SVD solver. Fitting all components exposes the complete eigenvalue spectrum before choosing a reduced dimension. Because the inputs are standardized, this PCA is equivalent to working from their correlation structure rather than allowing the original measurement units to determine importance.

In [34]:
from sklearn.decomposition import PCA

N_PCA_FEATURES = len(PCA_FEATURES)
pca_full = PCA(n_components=N_PCA_FEATURES, svd_solver='full')
pca_scores_array = pca_full.fit_transform(standardized_features)
component_names = [f'PC{number}' for number in range(1, N_PCA_FEATURES + 1)]
pca_scores = pd.DataFrame(
    pca_scores_array,
    columns=component_names,
    index=standardized_features.index,
)

explained_variance_ratio = pca_full.explained_variance_ratio_
cumulative_explained_variance_ratio = np.cumsum(explained_variance_ratio)
pca_variance_table = pd.DataFrame(
    {
        'component': component_names,
        'component_number': np.arange(1, N_PCA_FEATURES + 1),
        'eigenvalue': pca_full.explained_variance_,
        'explained_variance_ratio': explained_variance_ratio,
        'explained_variance_percent': 100 * explained_variance_ratio,
        'cumulative_explained_variance_ratio': cumulative_explained_variance_ratio,
        'cumulative_explained_variance_percent': (
            100 * cumulative_explained_variance_ratio
        ),
        'eigenvalue_above_one': pca_full.explained_variance_ > 1,
    }
)

assert pca_scores.shape == (len(wine_cleaned), N_PCA_FEATURES)
assert np.isfinite(pca_scores.to_numpy()).all()
assert np.all(np.diff(pca_full.explained_variance_) <= 0)
assert np.isclose(explained_variance_ratio.sum(), 1.0)
assert np.isclose(cumulative_explained_variance_ratio[-1], 1.0)
assert np.allclose(
    pca_full.components_ @ pca_full.components_.T,
    np.eye(N_PCA_FEATURES),
    atol=1e-12,
)
assert np.allclose(pca_full.mean_, 0.0, atol=1e-12)
assert np.allclose(
    pca_scores.var(ddof=1).to_numpy(),
    pca_full.explained_variance_,
)

full_reconstruction = pca_full.inverse_transform(pca_scores_array)
assert np.allclose(
    full_reconstruction, standardized_features.to_numpy(), atol=1e-10
)

pca_variance_table

,component,component_number,eigenvalue,explained_variance_ratio,explained_variance_percent,cumulative_explained_variance_ratio,cumulative_explained_variance_percent,eigenvalue_above_one
0,PC1,1,2.988937,0.271671,27.167050,0.271671,27.167050,True
1,PC2,2,2.475732,0.225024,22.502425,0.496695,49.669476,True
2,PC3,3,1.587581,0.144298,14.429840,0.640993,64.099315,True
3,PC4,4,0.953581,0.086673,8.667287,0.727666,72.766602,False
4,PC5,5,0.742375,0.067476,6.747594,0.795142,79.514196,False
5,PC6,6,0.627156,0.057003,5.700345,0.852145,85.214541,False
6,PC7,7,0.521138,0.047367,4.736727,0.899513,89.951268,False
7,PC8,8,0.507694,0.046145,4.614532,0.945658,94.565801,False
8,PC9,9,0.336784,0.030611,3.061100,0.976269,97.626900,False
9,PC10,10,0.226001,0.020542,2.054165,0.996811,99.681066,False


In [35]:
variance_thresholds = [0.80, 0.90, 0.95]
threshold_rows = []
for threshold in variance_thresholds:
    component_index = int(
        np.searchsorted(cumulative_explained_variance_ratio, threshold, side='left')
    )
    threshold_rows.append(
        {
            'target_variance_ratio': threshold,
            'target_variance_percent': 100 * threshold,
            'minimum_components': component_index + 1,
            'achieved_variance_ratio': (
                cumulative_explained_variance_ratio[component_index]
            ),
            'achieved_variance_percent': (
                100 * cumulative_explained_variance_ratio[component_index]
            ),
        }
    )

pca_threshold_table = pd.DataFrame(threshold_rows)
assert pca_threshold_table['minimum_components'].is_monotonic_increasing
assert (
    pca_threshold_table['achieved_variance_ratio']
    >= pca_threshold_table['target_variance_ratio']
).all()

kaiser_component_count = int((pca_full.explained_variance_ > 1).sum())
kaiser_cumulative_ratio = float(
    cumulative_explained_variance_ratio[kaiser_component_count - 1]
)
PCA_SELECTION_THRESHOLD = 0.90
SELECTED_N_COMPONENTS = int(
    pca_threshold_table.loc[
        pca_threshold_table['target_variance_ratio'].eq(PCA_SELECTION_THRESHOLD),
        'minimum_components',
    ].iloc[0]
)

print(
    f'PC1 explains {100 * explained_variance_ratio[0]:.2f}% and '
    f'PC{N_PCA_FEATURES} explains {100 * explained_variance_ratio[-1]:.2f}%.'
)
print(
    f'Kaiser criterion retains {kaiser_component_count} components '
    f'({100 * kaiser_cumulative_ratio:.2f}% cumulative variance).'
)
print(
    f'Selected for the 90% rule: {SELECTED_N_COMPONENTS} components.'
)
pca_threshold_table

PC1 explains 27.17% and PC11 explains 0.32%.
Kaiser criterion retains 3 components (64.10% cumulative variance).
Selected for the 90% rule: 8 components.


,target_variance_ratio,target_variance_percent,minimum_components,achieved_variance_ratio,achieved_variance_percent
0,0.80,80.0,6,0.852145,85.214541
1,0.90,90.0,8,0.945658,94.565801
2,0.95,95.0,9,0.976269,97.626900


### Component-selection rule

The cumulative explained-variance threshold is the primary selection rule because it directly states how much standardized feature variance is retained. A 90% target is used as the main reduced representation, while 80% and 95% are retained for sensitivity comparisons. The Kaiser rule (eigenvalue greater than 1) is also reported as a common heuristic, but it can preserve substantially less total variance. PC1 has the highest explained variance by construction; the last component has the lowest and represents a nearly redundant direction.

In [36]:
pca_scores_with_labels = pca_scores.copy()
pca_scores_with_labels[TARGET_COLUMN] = wine_cleaned[TARGET_COLUMN].to_numpy()
pca_scores_with_labels[LABEL_COLUMN] = wine_cleaned[LABEL_COLUMN].to_numpy()

PCA_SCORES_PATH = PROCESSED_DIR / 'wine_pca_scores_all.csv'
PCA_VARIANCE_PATH = TABLES_DIR / 'wine_pca_explained_variance.csv'
PCA_THRESHOLDS_PATH = TABLES_DIR / 'wine_pca_component_thresholds.csv'

pca_scores_with_labels.to_csv(PCA_SCORES_PATH, index=False)
pca_variance_table.to_csv(PCA_VARIANCE_PATH, index=False)
pca_threshold_table.to_csv(PCA_THRESHOLDS_PATH, index=False)

pca_scores_round_trip = pd.read_csv(PCA_SCORES_PATH)
assert pca_scores_round_trip.shape == pca_scores_with_labels.shape
assert pca_scores_round_trip[TARGET_COLUMN].equals(wine_cleaned[TARGET_COLUMN])
assert pca_scores_round_trip[LABEL_COLUMN].equals(wine_cleaned[LABEL_COLUMN])
assert np.allclose(
    pca_scores_round_trip[component_names].to_numpy(),
    pca_scores.to_numpy(),
)

print(f'Saved full PCA scores to: {PCA_SCORES_PATH}')
print(f'Saved explained-variance table to: {PCA_VARIANCE_PATH}')
print(f'Saved component thresholds to: {PCA_THRESHOLDS_PATH}')

Saved full PCA scores to: D:\ml2\lab1_ml2\data\processed\wine_pca_scores_all.csv
Saved explained-variance table to: D:\ml2\lab1_ml2\report\tables\wine_pca_explained_variance.csv
Saved component thresholds to: D:\ml2\lab1_ml2\report\tables\wine_pca_component_thresholds.csv


In [37]:
component_numbers = np.arange(1, N_PCA_FEATURES + 1)
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

axes[0].bar(
    component_numbers,
    100 * explained_variance_ratio,
    color='#4c78a8',
    edgecolor='white',
)
axes[0].axhline(
    100 / N_PCA_FEATURES,
    color='#666666',
    linestyle='--',
    linewidth=1.2,
    label='Average variance per component',
)
axes[0].set_ylabel('Explained variance (%)')
axes[0].set_title('Individual explained variance')
axes[0].legend(loc='upper right')

axes[1].plot(
    component_numbers,
    100 * cumulative_explained_variance_ratio,
    color='#d95f02',
    marker='o',
    linewidth=2,
)
threshold_colors = {0.80: '#7f7f7f', 0.90: '#2ca02c', 0.95: '#9467bd'}
for row in pca_threshold_table.itertuples(index=False):
    threshold = row.target_variance_ratio
    color = threshold_colors[threshold]
    axes[1].axhline(
        100 * threshold,
        color=color,
        linestyle='--',
        linewidth=1,
        label=f'{int(100 * threshold)}% target: {row.minimum_components} PCs',
    )
    axes[1].scatter(
        row.minimum_components,
        row.achieved_variance_percent,
        color=color,
        s=45,
        zorder=3,
    )

axes[1].set_xlabel('Number of principal components')
axes[1].set_ylabel('Cumulative explained variance (%)')
axes[1].set_title('Cumulative explained variance')
axes[1].set_xticks(component_numbers)
axes[1].set_ylim(0, 105)
axes[1].legend(loc='lower right')
fig.suptitle('PCA explained variance after feature standardization', y=1.01)
fig.tight_layout()

PCA_SCREE_PATH = FIGURES_DIR / 'wine_pca_explained_variance.png'
fig.savefig(PCA_SCREE_PATH, dpi=180, bbox_inches='tight')
plt.close(fig)
print(f'Saved PCA explained-variance figure to: {PCA_SCREE_PATH}')

Saved PCA explained-variance figure to: D:\ml2\lab1_ml2\report\figures\wine_pca_explained_variance.png


## 12. PC1/PC2 interpretation and labelled projections

Two complementary quantities are reported for interpretation. The **eigenvector coefficient** is the weight used to construct a component from the standardized features. The **correlation loading** is the Pearson correlation between an original standardized feature and a component score, which is easier to compare across components with different eigenvalues. Features are ranked by the absolute correlation loading, while the sign indicates direction within this fitted solution.

In [38]:
component_coefficients = pd.DataFrame(
    pca_full.components_.T,
    index=PCA_FEATURES,
    columns=component_names,
)
component_coefficients.index.name = 'feature'

combined_for_correlations = np.column_stack(
    [standardized_features.to_numpy(), pca_scores.to_numpy()]
)
combined_correlation = np.corrcoef(combined_for_correlations, rowvar=False)
correlation_loadings = pd.DataFrame(
    combined_correlation[:N_PCA_FEATURES, N_PCA_FEATURES:],
    index=PCA_FEATURES,
    columns=component_names,
)
correlation_loadings.index.name = 'feature'

feature_sample_standard_deviation = (
    standardized_features.std(ddof=1).to_numpy()[:, np.newaxis]
)
correlation_loadings_from_pca = (
    pca_full.components_.T
    * np.sqrt(pca_full.explained_variance_)[np.newaxis, :]
    / feature_sample_standard_deviation
)

assert component_coefficients.shape == (N_PCA_FEATURES, N_PCA_FEATURES)
assert correlation_loadings.shape == (N_PCA_FEATURES, N_PCA_FEATURES)
assert np.isfinite(correlation_loadings.to_numpy()).all()
assert (correlation_loadings.abs().to_numpy() <= 1 + 1e-12).all()
assert np.allclose(
    correlation_loadings.to_numpy(),
    correlation_loadings_from_pca,
    atol=1e-12,
)
assert np.allclose(
    correlation_loadings.pow(2).sum(axis=1).to_numpy(),
    np.ones(N_PCA_FEATURES),
    atol=1e-12,
)

display(correlation_loadings[['PC1', 'PC2']].round(4))

,PC1,PC2
feature,,
fixed acidity,-0.4751,0.4824
volatile acidity,-0.6737,0.1094
citric acid,0.2317,0.3452
residual sugar,0.5144,0.5790
chlorides,-0.5524,0.4325
free sulfur dioxide,0.7360,0.2221
total sulfur dioxide,0.8233,0.2552
density,-0.2361,0.8890
pH,-0.3248,-0.2948


In [39]:
pc1_pc2_ranking_parts = []
for component in ['PC1', 'PC2']:
    component_ranking = pd.DataFrame(
        {
            'feature': PCA_FEATURES,
            'eigenvector_coefficient': component_coefficients[component].to_numpy(),
            'absolute_eigenvector_coefficient': component_coefficients[component].abs().to_numpy(),
            'correlation_loading': correlation_loadings[component].to_numpy(),
            'absolute_correlation_loading': correlation_loadings[component].abs().to_numpy(),
        }
    ).sort_values(
        ['absolute_correlation_loading', 'feature'],
        ascending=[False, True],
        kind='stable',
    )
    component_ranking.insert(0, 'rank', np.arange(1, N_PCA_FEATURES + 1))
    component_ranking.insert(0, 'component', component)
    pc1_pc2_ranking_parts.append(component_ranking)

pc1_pc2_feature_ranking = pd.concat(pc1_pc2_ranking_parts, ignore_index=True)
assert pc1_pc2_feature_ranking.shape == (2 * N_PCA_FEATURES, 7)
assert (
    pc1_pc2_feature_ranking.groupby('component')['feature'].nunique()
    == N_PCA_FEATURES
).all()

PCA_COEFFICIENTS_PATH = TABLES_DIR / 'wine_pca_component_coefficients.csv'
PCA_CORRELATION_LOADINGS_PATH = TABLES_DIR / 'wine_pca_correlation_loadings.csv'
PCA_PC1_PC2_RANKING_PATH = TABLES_DIR / 'wine_pca_pc1_pc2_feature_ranking.csv'

component_coefficients.reset_index().to_csv(PCA_COEFFICIENTS_PATH, index=False)
correlation_loadings.reset_index().to_csv(
    PCA_CORRELATION_LOADINGS_PATH, index=False
)
pc1_pc2_feature_ranking.to_csv(PCA_PC1_PC2_RANKING_PATH, index=False)

coefficient_round_trip = pd.read_csv(PCA_COEFFICIENTS_PATH)
loading_round_trip = pd.read_csv(PCA_CORRELATION_LOADINGS_PATH)
ranking_round_trip = pd.read_csv(PCA_PC1_PC2_RANKING_PATH)
assert coefficient_round_trip['feature'].tolist() == PCA_FEATURES
assert loading_round_trip['feature'].tolist() == PCA_FEATURES
assert np.allclose(
    coefficient_round_trip[component_names].to_numpy(),
    component_coefficients.to_numpy(),
)
assert np.allclose(
    loading_round_trip[component_names].to_numpy(),
    correlation_loadings.to_numpy(),
)
assert ranking_round_trip.shape == pc1_pc2_feature_ranking.shape

display(pc1_pc2_feature_ranking.groupby('component', sort=False).head(5).round(4))
print(f'Saved component coefficients to: {PCA_COEFFICIENTS_PATH}')
print(f'Saved correlation loadings to: {PCA_CORRELATION_LOADINGS_PATH}')
print(f'Saved PC1/PC2 feature ranking to: {PCA_PC1_PC2_RANKING_PATH}')

,component,rank,feature,eigenvector_coefficient,absolute_eigenvector_coefficient,correlation_loading,absolute_correlation_loading
0,PC1,1,total sulfur dioxide,0.4763,0.4763,0.8233,0.8233
1,PC1,2,free sulfur dioxide,0.4258,0.4258,0.7360,0.7360
2,PC1,3,volatile acidity,-0.3897,0.3897,-0.6737,0.6737
3,PC1,4,chlorides,-0.3196,0.3196,-0.5524,0.5524
4,PC1,5,sulphates,-0.3176,0.3176,-0.5490,0.5490
11,PC2,1,density,0.5651,0.5651,0.8890,0.8890
12,PC2,2,alcohol,-0.4652,0.4652,-0.7320,0.7320
13,PC2,3,residual sugar,0.3680,0.3680,0.5790,0.5790
14,PC2,4,fixed acidity,0.3066,0.3066,0.4824,0.4824
15,PC2,5,chlorides,0.2749,0.2749,0.4325,0.4325


Saved component coefficients to: D:\ml2\lab1_ml2\report\tables\wine_pca_component_coefficients.csv
Saved correlation loadings to: D:\ml2\lab1_ml2\report\tables\wine_pca_correlation_loadings.csv
Saved PC1/PC2 feature ranking to: D:\ml2\lab1_ml2\report\tables\wine_pca_pc1_pc2_feature_ranking.csv


### Interpretation of the first two components

- **PC1 (27.17%)** is most strongly associated with total sulfur dioxide (+0.823), free sulfur dioxide (+0.736), volatile acidity (-0.674), chlorides (-0.552), sulphates (-0.549), and residual sugar (+0.514). Higher PC1 scores therefore describe samples with more sulfur dioxide and residual sugar but lower volatile acidity, chlorides, and sulphates within this fitted dataset.
- **PC2 (22.50%)** is dominated by density (+0.889), alcohol (-0.732), residual sugar (+0.579), fixed acidity (+0.482), and chlorides (+0.432). Higher PC2 scores mainly represent denser, sweeter, lower-alcohol wines.

The signs of PCA axes are arbitrary: multiplying one component and all its scores by -1 gives the same solution. Interpretation should therefore emphasize the relative magnitudes and relationships among signs, not treat a positive sign as intrinsically better. PC1 and PC2 together retain 49.67% of standardized variance, so the two-dimensional projections below are informative but incomplete.

In [40]:
assert pca_scores_with_labels.index.equals(wine_cleaned.index)
assert pca_scores_with_labels[TARGET_COLUMN].equals(wine_cleaned[TARGET_COLUMN])
assert pca_scores_with_labels[LABEL_COLUMN].equals(wine_cleaned[LABEL_COLUMN])

pc1_variance_percent = 100 * explained_variance_ratio[0]
pc2_variance_percent = 100 * explained_variance_ratio[1]
quality_levels = sorted(pca_scores_with_labels[TARGET_COLUMN].unique())
wine_type_palette = {'white': '#4c78a8', 'red': '#b2182b'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

for wine_type in ['white', 'red']:
    type_mask = pca_scores_with_labels[LABEL_COLUMN].eq(wine_type)
    type_count = int(type_mask.sum())
    axes[0].scatter(
        pca_scores_with_labels.loc[type_mask, 'PC1'],
        pca_scores_with_labels.loc[type_mask, 'PC2'],
        s=18,
        alpha=0.45,
        color=wine_type_palette[wine_type],
        edgecolors='none',
        label=f'{wine_type.capitalize()} (n={type_count:,})',
        rasterized=True,
    )
axes[0].set_title('Projection labelled by wine type')
axes[0].legend(loc='upper left', frameon=True)

quality_scatter = axes[1].scatter(
    pca_scores_with_labels['PC1'],
    pca_scores_with_labels['PC2'],
    c=pca_scores_with_labels[TARGET_COLUMN],
    cmap='viridis',
    vmin=min(quality_levels),
    vmax=max(quality_levels),
    s=18,
    alpha=0.55,
    edgecolors='none',
    rasterized=True,
)
axes[1].set_title('Projection labelled by quality score')
quality_colorbar = fig.colorbar(quality_scatter, ax=axes[1], pad=0.02)
quality_colorbar.set_label('Quality score')
quality_colorbar.set_ticks(quality_levels)

for axis in axes:
    axis.axhline(0, color='#999999', linewidth=0.8, linestyle='--', zorder=0)
    axis.axvline(0, color='#999999', linewidth=0.8, linestyle='--', zorder=0)
    axis.set_xlabel(f'PC1 ({pc1_variance_percent:.2f}%)')
    axis.set_ylabel(f'PC2 ({pc2_variance_percent:.2f}%)')
    axis.grid(True, color='#e6e6e6', linewidth=0.6)

fig.suptitle(
    'Wine Quality observations projected onto PC1 and PC2 '    f'({pc1_variance_percent + pc2_variance_percent:.2f}% cumulative variance)',
    y=1.02,
)
fig.tight_layout()

PCA_PC1_PC2_LABELS_PATH = FIGURES_DIR / 'wine_pca_pc1_pc2_by_labels.png'
fig.savefig(PCA_PC1_PC2_LABELS_PATH, dpi=180, bbox_inches='tight')
plt.close(fig)
print(f'Saved PC1/PC2 labelled projections to: {PCA_PC1_PC2_LABELS_PATH}')

Saved PC1/PC2 labelled projections to: D:\ml2\lab1_ml2\report\figures\wine_pca_pc1_pc2_by_labels.png


### Projection cautions

The wine-type projection shows substantial separation along PC1: red wines are concentrated at negative PC1 scores, while most white wines lie closer to zero or on the positive side. The groups still overlap near the boundary, so PC1/PC2 do not provide perfect type separation. In contrast, the quality colors overlap strongly and do not form distinct clusters in this two-dimensional view. These axes preserve less than half of the total variance, so later components may contain additional structure. Quality classes are also highly imbalanced (only 5 cleaned observations have quality 9), so rare colors must not be interpreted as stable clusters. The plots retain every cleaned observation and the full PC1/PC2 range; the isolated extreme scores reflect the valid extreme measurements retained during cleaning.

## 13. Component-count sensitivity and reconstruction

PCA models are refitted with 2, 3, 6, 8, and 9 components. Each reduced score matrix is mapped back to the 11-feature standardized space with `inverse_transform`. Reconstruction error is therefore measured against the standardized inputs, where every original feature has population variance 1. Lower MSE, RMSE, MAE, and relative Frobenius error indicate a closer reconstruction.

In [41]:
TESTED_COMPONENT_COUNTS = [2, 3, 6, 8, 9]
SELECTED_COMPONENT_COUNT = 8
standardized_matrix = standardized_features.to_numpy()
centered_standardized_matrix = (
    standardized_matrix - standardized_matrix.mean(axis=0, keepdims=True)
)
reconstructed_by_component_count = {}
reconstruction_rows = []

for component_count in TESTED_COMPONENT_COUNTS:
    reduced_pca = PCA(n_components=component_count, svd_solver='full')
    reduced_scores = reduced_pca.fit_transform(standardized_matrix)
    reconstructed_matrix = reduced_pca.inverse_transform(reduced_scores)
    reconstruction_difference = standardized_matrix - reconstructed_matrix
    retained_variance_ratio = float(
        reduced_pca.explained_variance_ratio_.sum()
    )
    reconstruction_mse = float(np.mean(reconstruction_difference ** 2))
    reconstruction_rmse = float(np.sqrt(reconstruction_mse))
    reconstruction_mae = float(np.mean(np.abs(reconstruction_difference)))
    relative_frobenius_error = float(
        np.linalg.norm(reconstruction_difference, ord='fro')
        / np.linalg.norm(centered_standardized_matrix, ord='fro')
    )

    reconstructed_by_component_count[component_count] = reconstructed_matrix
    reconstruction_rows.append(
        {
            'n_components': component_count,
            'retained_variance_ratio': retained_variance_ratio,
            'retained_variance_percent': 100 * retained_variance_ratio,
            'discarded_variance_ratio': 1 - retained_variance_ratio,
            'discarded_variance_percent': 100 * (1 - retained_variance_ratio),
            'reconstruction_mse': reconstruction_mse,
            'reconstruction_rmse': reconstruction_rmse,
            'reconstruction_mae': reconstruction_mae,
            'relative_frobenius_error': relative_frobenius_error,
            'score_dimension_ratio': component_count / N_PCA_FEATURES,
            'feature_dimension_reduction_percent': (
                100 * (1 - component_count / N_PCA_FEATURES)
            ),
        }
    )

pca_reconstruction_summary = pd.DataFrame(reconstruction_rows)
expected_retained_ratios = np.array(
    [cumulative_explained_variance_ratio[k - 1] for k in TESTED_COMPONENT_COUNTS]
)

assert pca_reconstruction_summary.shape == (len(TESTED_COMPONENT_COUNTS), 11)
assert np.allclose(
    pca_reconstruction_summary['retained_variance_ratio'],
    expected_retained_ratios,
    atol=1e-12,
)
assert np.allclose(
    pca_reconstruction_summary['reconstruction_mse'],
    pca_reconstruction_summary['discarded_variance_ratio'],
    atol=1e-12,
)
assert np.allclose(
    pca_reconstruction_summary['reconstruction_rmse'],
    pca_reconstruction_summary['relative_frobenius_error'],
    atol=1e-12,
)
assert (np.diff(pca_reconstruction_summary['retained_variance_ratio']) > 0).all()
assert (np.diff(pca_reconstruction_summary['reconstruction_mse']) < 0).all()
assert all(
    reconstructed.shape == standardized_matrix.shape
    for reconstructed in reconstructed_by_component_count.values()
)

display(pca_reconstruction_summary.round(6))

,n_components,retained_variance_ratio,retained_variance_percent,discarded_variance_ratio,discarded_variance_percent,reconstruction_mse,reconstruction_rmse,reconstruction_mae,relative_frobenius_error,score_dimension_ratio,feature_dimension_reduction_percent
0,2,0.496695,49.669476,0.503305,50.330524,0.503305,0.709440,0.513568,0.709440,0.181818,81.818182
1,3,0.640993,64.099315,0.359007,35.900685,0.359007,0.599172,0.428110,0.599172,0.272727,72.727273
2,6,0.852145,85.214541,0.147855,14.785459,0.147855,0.384519,0.280358,0.384519,0.545455,45.454545
3,8,0.945658,94.565801,0.054342,5.434199,0.054342,0.233114,0.162438,0.233114,0.727273,27.272727
4,9,0.976269,97.626900,0.023731,2.373100,0.023731,0.154049,0.100620,0.154049,0.818182,18.181818


In [42]:
selected_reconstruction = reconstructed_by_component_count[SELECTED_COMPONENT_COUNT]
selected_difference = standardized_matrix - selected_reconstruction
selected_original_frame = pd.DataFrame(standardized_matrix, columns=PCA_FEATURES)
selected_reconstruction_frame = pd.DataFrame(
    selected_reconstruction, columns=PCA_FEATURES
)

pca_reconstruction_by_feature = pd.DataFrame(
    {
        'feature': PCA_FEATURES,
        'original_mean': selected_original_frame.mean().to_numpy(),
        'reconstructed_mean': selected_reconstruction_frame.mean().to_numpy(),
        'original_population_std': selected_original_frame.std(ddof=0).to_numpy(),
        'reconstructed_population_std': selected_reconstruction_frame.std(ddof=0).to_numpy(),
        'reconstruction_mse': np.mean(selected_difference ** 2, axis=0),
        'reconstruction_rmse': np.sqrt(np.mean(selected_difference ** 2, axis=0)),
        'reconstruction_mae': np.mean(np.abs(selected_difference), axis=0),
    }
).sort_values(
    ['reconstruction_rmse', 'feature'],
    ascending=[False, True],
    kind='stable',
).reset_index(drop=True)
pca_reconstruction_by_feature.insert(
    0, 'rank_by_rmse', np.arange(1, N_PCA_FEATURES + 1)
)

assert pca_reconstruction_by_feature.shape == (N_PCA_FEATURES, 9)
assert np.allclose(pca_reconstruction_by_feature['original_mean'], 0.0, atol=1e-12)
assert np.allclose(
    pca_reconstruction_by_feature['original_population_std'], 1.0, atol=1e-12
)
assert np.isclose(
    pca_reconstruction_by_feature['reconstruction_mse'].mean(),
    pca_reconstruction_summary.loc[
        pca_reconstruction_summary['n_components'].eq(SELECTED_COMPONENT_COUNT),
        'reconstruction_mse',
    ].item(),
    atol=1e-12,
)

PCA_RECONSTRUCTION_SUMMARY_PATH = (
    TABLES_DIR / 'wine_pca_reconstruction_by_components.csv'
)
PCA_RECONSTRUCTION_FEATURE_PATH = (
    TABLES_DIR / 'wine_pca_reconstruction_by_feature_k8.csv'
)
pca_reconstruction_summary.to_csv(PCA_RECONSTRUCTION_SUMMARY_PATH, index=False)
pca_reconstruction_by_feature.to_csv(PCA_RECONSTRUCTION_FEATURE_PATH, index=False)

reconstruction_summary_round_trip = pd.read_csv(
    PCA_RECONSTRUCTION_SUMMARY_PATH
)
reconstruction_feature_round_trip = pd.read_csv(
    PCA_RECONSTRUCTION_FEATURE_PATH
)
assert reconstruction_summary_round_trip['n_components'].tolist() == (
    TESTED_COMPONENT_COUNTS
)
assert reconstruction_feature_round_trip['feature'].tolist() == (
    pca_reconstruction_by_feature['feature'].tolist()
)
assert np.allclose(
    reconstruction_summary_round_trip.select_dtypes(include='number'),
    pca_reconstruction_summary.select_dtypes(include='number'),
)
assert np.allclose(
    reconstruction_feature_round_trip.select_dtypes(include='number'),
    pca_reconstruction_by_feature.select_dtypes(include='number'),
)

display(pca_reconstruction_by_feature.round(6))
print(f'Saved component sensitivity table to: {PCA_RECONSTRUCTION_SUMMARY_PATH}')
print(f'Saved k=8 feature reconstruction table to: {PCA_RECONSTRUCTION_FEATURE_PATH}')

,rank_by_rmse,feature,original_mean,reconstructed_mean,original_population_std,reconstructed_population_std,reconstruction_mse,reconstruction_rmse,reconstruction_mae
0,1,total sulfur dioxide,-0.0,-0.0,1.0,0.920997,0.151764,0.389569,0.296807
1,2,free sulfur dioxide,0.0,0.0,1.0,0.949450,0.098544,0.313917,0.239533
2,3,volatile acidity,-0.0,-0.0,1.0,0.954338,0.089238,0.298728,0.227037
3,4,citric acid,0.0,0.0,1.0,0.963257,0.072136,0.268581,0.204406
4,5,fixed acidity,0.0,0.0,1.0,0.969105,0.060836,0.246650,0.188520
5,6,chlorides,0.0,0.0,1.0,0.980234,0.039141,0.197842,0.150749
6,7,alcohol,0.0,0.0,1.0,0.984807,0.030154,0.173650,0.131303
7,8,density,0.0,0.0,1.0,0.988764,0.022345,0.149483,0.109539
8,9,sulphates,-0.0,-0.0,1.0,0.992976,0.013999,0.118317,0.089838
9,10,residual sugar,0.0,-0.0,1.0,0.994956,0.010062,0.100311,0.074397


Saved component sensitivity table to: D:\ml2\lab1_ml2\report\tables\wine_pca_reconstruction_by_components.csv
Saved k=8 feature reconstruction table to: D:\ml2\lab1_ml2\report\tables\wine_pca_reconstruction_by_feature_k8.csv


In [43]:
selected_summary_row = pca_reconstruction_summary.loc[
    pca_reconstruction_summary['n_components'].eq(SELECTED_COMPONENT_COUNT)
].iloc[0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
component_counts = pca_reconstruction_summary['n_components']

axes[0].plot(
    component_counts,
    pca_reconstruction_summary['retained_variance_percent'],
    color='#4c78a8',
    marker='o',
    linewidth=2,
)
axes[0].axhline(90, color='#2ca02c', linestyle='--', linewidth=1.2, label='90% target')
axes[0].scatter(
    [SELECTED_COMPONENT_COUNT],
    [selected_summary_row['retained_variance_percent']],
    color='#d95f02',
    s=70,
    zorder=3,
    label='Selected k=8',
)
axes[0].set_title('Variance retained as component count increases')
axes[0].set_xlabel('Number of components')
axes[0].set_ylabel('Retained variance (%)')
axes[0].set_xticks(TESTED_COMPONENT_COUNTS)
axes[0].set_ylim(0, 102)
axes[0].legend(loc='lower right')

axes[1].plot(
    component_counts,
    pca_reconstruction_summary['reconstruction_rmse'],
    color='#b2182b',
    marker='o',
    linewidth=2,
)
axes[1].scatter(
    [SELECTED_COMPONENT_COUNT],
    [selected_summary_row['reconstruction_rmse']],
    color='#d95f02',
    s=70,
    zorder=3,
)
axes[1].set_title('Reconstruction error falls with more components')
axes[1].set_xlabel('Number of components')
axes[1].set_ylabel('RMSE in standardized feature units')
axes[1].set_xticks(TESTED_COMPONENT_COUNTS)
axes[1].set_ylim(bottom=0)

original_flat = standardized_matrix.ravel()
reconstructed_flat = selected_reconstruction.ravel()
comparison_minimum = float(min(original_flat.min(), reconstructed_flat.min()))
comparison_maximum = float(max(original_flat.max(), reconstructed_flat.max()))
comparison_padding = 0.03 * (comparison_maximum - comparison_minimum)
comparison_limits = [
    comparison_minimum - comparison_padding,
    comparison_maximum + comparison_padding,
]
hexbin = axes[2].hexbin(
    original_flat,
    reconstructed_flat,
    gridsize=65,
    bins='log',
    mincnt=1,
    cmap='Blues',
)
axes[2].plot(
    comparison_limits,
    comparison_limits,
    color='#d95f02',
    linestyle='--',
    linewidth=1.3,
    label='Perfect reconstruction',
)
axes[2].set_xlim(comparison_limits)
axes[2].set_ylim(comparison_limits)
axes[2].set_aspect('equal', adjustable='box')
axes[2].set_title('Original versus k=8 reconstruction')
axes[2].set_xlabel('Original standardized value')
axes[2].set_ylabel('Reconstructed standardized value')
axes[2].legend(loc='upper left')
comparison_colorbar = fig.colorbar(hexbin, ax=axes[2], pad=0.02)
comparison_colorbar.set_label('Observation-feature pairs per hexagon')

for axis in axes[:2]:
    axis.grid(True, color='#e6e6e6', linewidth=0.6)

fig.suptitle('PCA component-count sensitivity and reconstruction', y=1.02)
fig.tight_layout()

PCA_RECONSTRUCTION_FIGURE_PATH = (
    FIGURES_DIR / 'wine_pca_component_sensitivity_and_reconstruction.png'
)
fig.savefig(PCA_RECONSTRUCTION_FIGURE_PATH, dpi=180, bbox_inches='tight')
plt.close(fig)
print(f'Saved PCA reconstruction figure to: {PCA_RECONSTRUCTION_FIGURE_PATH}')

Saved PCA reconstruction figure to: D:\ml2\lab1_ml2\report\figures\wine_pca_component_sensitivity_and_reconstruction.png


### Component-count comparison and selection

- With **2 components**, PCA retains 49.67% of variance and reconstruction RMSE is 0.709 standardized units. With **3 components**, retention rises to 64.10% and RMSE falls to 0.599. These compact projections are useful for visualization but discard too much information for the main reduced dataset.
- With **6 components**, PCA retains 85.21% and RMSE is 0.385. This is below the predefined 90% target.
- With **8 components**, PCA retains 94.57% and RMSE falls to 0.233 while reducing the score dimension from 11 to 8 features (27.27% fewer dimensions). It is the smallest tested configuration that clears the 90% target and remains the main selection.
- With **9 components**, retention reaches 97.63% and RMSE is 0.154. The ninth component adds 3.06 percentage points of retained variance, but it also gives up part of the dimensionality reduction.

For the selected k=8 reconstruction, the largest feature-level RMSE values occur for total sulfur dioxide (0.390), free sulfur dioxide (0.314), and volatile acidity (0.299). The comparison is in standardized units; reconstruction in original measurement units would additionally require the scaler's inverse transformation.